**Url del dataset:** https://archive.ics.uci.edu/dataset/545/rice+cammeo+and+osmancik

# Punto 1 — Kernel cuadrático

**Inteligencia Artificial — Kernels y SVM**

Se estudia el kernel
$$
K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}^\top \mathbf{z} + c)^2 = \varphi(\mathbf{x})^\top \varphi(\mathbf{z}),
$$
con $c=9$ y $\mathbf{x}, \mathbf{z}\in\mathbb{R}^2$.

El notebook deduce $\varphi$, la implementa para cualquier dimensión $d$, resuelve los incisos a) y b), verifica la identidad de forma general y analiza el costo del inciso c). El código del punto 1 usa solamente `numpy`, `itertools` y `time`.

## 1. Deducción de $\varphi$

Partimos de
$$
(\mathbf{x}^\top \mathbf{z} + c)^2
= \left(\sum_{i=1}^{d} x_i z_i + c\right)^2.
$$

Al desarrollar el cuadrado se obtienen cuatro tipos de términos:
$$
(\mathbf{x}^\top \mathbf{z} + c)^2
= \sum_{i=1}^{d} x_i^2 z_i^2
+ 2\sum_{1\le i<j\le d} x_i x_j z_i z_j
+ 2c\sum_{i=1}^{d} x_i z_i
+ c^2.
$$

Cada término se puede separar en un factor que depende de $\mathbf{x}$ y otro que depende de $\mathbf{z}$:

- $x_i^2 z_i^2 = (x_i^2)(z_i^2)$.
- $2x_i x_j z_i z_j = (\sqrt{2}\,x_i x_j)(\sqrt{2}\,z_i z_j)$.
- $2c\,x_i z_i = (\sqrt{2c}\,x_i)(\sqrt{2c}\,z_i)$.
- $c^2 = c\cdot c$.

Por lo tanto, una elección posible de $\varphi$ es
$$
\varphi(\mathbf{x}) =
\big[
\underbrace{x_1^2,\ldots,x_d^2}_{d\text{ cuadrados}},\;
\underbrace{\sqrt{2}\,x_i x_j\ (i<j)}_{d(d-1)/2\text{ cruzados}},\;
\underbrace{\sqrt{2c}\,x_1,\ldots,\sqrt{2c}\,x_d}_{d\text{ lineales}},\;
\underbrace{c}_{1\text{ constante}}
\big].
$$

Los pares $i<j$ se ordenan de forma lexicográfica: $(1,2),(1,3),\ldots,(d-1,d)$. Para $d=2$ queda
$$
\varphi(\mathbf{x}) =
[x_1^2,\; x_2^2,\; \sqrt{2}\,x_1x_2,\; \sqrt{2c}\,x_1,\; \sqrt{2c}\,x_2,\; c].
$$

> **Nota:** se necesita $c\ge 0$ para que $\sqrt{2c}$ sea real. En este punto $c=9$.

## 2. Implementación de $\varphi$ para cualquier dimensión

La función recibe un vector `x` y el valor `c`, y devuelve las componentes en el mismo orden de la deducción anterior. Se usa `itertools.combinations` para recorrer los pares $i<j$ sin repetirlos.

In [1]:
import itertools
import time

import numpy as np


def phi(x, c):
    """Devuelve φ(x) para el kernel K(x, z) = (x·z + c)²."""
    x = np.asarray(x, dtype=float)  # aseguramos un array de números reales
    d = x.size                      # dimensión del vector original

    # 1) Cuadrados: x₁², ..., x_d²
    cuadrados = x ** 2

    # 2) Términos cruzados: √2·x_i·x_j para cada par i < j.
    #    combinations(range(d), 2) genera (0,1), (0,2), ... en orden lexicográfico.
    cruzados = np.array(
        [np.sqrt(2.0) * x[i] * x[j] for i, j in itertools.combinations(range(d), 2)],
        dtype=float,
    )

    # 3) Términos lineales: √(2c)·x₁, ..., √(2c)·x_d
    lineales = np.sqrt(2.0 * c) * x

    # 4) Término constante: c
    constante = np.array([c], dtype=float)

    # El orden es siempre: cuadrados, cruzados, lineales, constante.
    return np.concatenate([cuadrados, cruzados, lineales, constante])

## 3. Punto a) — Cálculo de $\varphi([2,-3]^\top)$

Con $c=9$ y $\mathbf{x}=[2,-3]^\top$, calculamos el vector numérico usando la función anterior.

In [2]:
c = 9
x = np.array([2.0, -3.0])

phi_x = phi(x, c)

print("x =", x)
print("φ(x) =", np.round(phi_x, 4))
print()
print("Componentes de φ(x):")
print(f"  x₁²        = {phi_x[0]:.4f}")
print(f"  x₂²        = {phi_x[1]:.4f}")
print(f"  √2·x₁·x₂   = {phi_x[2]:.4f}")
print(f"  √(2c)·x₁   = {phi_x[3]:.4f}")
print(f"  √(2c)·x₂   = {phi_x[4]:.4f}")
print(f"  c          = {phi_x[5]:.4f}")

x = [ 2. -3.]
φ(x) = [  4.       9.      -8.4853   8.4853 -12.7279   9.    ]

Componentes de φ(x):
  x₁²        = 4.0000
  x₂²        = 9.0000
  √2·x₁·x₂   = -8.4853
  √(2c)·x₁   = 8.4853
  √(2c)·x₂   = -12.7279
  c          = 9.0000


**Forma exacta.** Como $\sqrt{2c}=\sqrt{18}=3\sqrt{2}$:

- $x_1^2 = 2^2 = 4$
- $x_2^2 = (-3)^2 = 9$
- $\sqrt{2}\,x_1x_2 = \sqrt{2}\cdot 2\cdot(-3) = -6\sqrt{2}$
- $\sqrt{2c}\,x_1 = 3\sqrt{2}\cdot 2 = 6\sqrt{2}$
- $\sqrt{2c}\,x_2 = 3\sqrt{2}\cdot(-3) = -9\sqrt{2}$
- $c = 9$

Por lo tanto,
$$
\varphi([2,-3]^\top) = [4,\; 9,\; -6\sqrt{2},\; 6\sqrt{2},\; -9\sqrt{2},\; 9]
\approx [4,\; 9,\; -8.4853,\; 8.4853,\; -12.7279,\; 9].
$$

## 4. Punto b) — Producto $\varphi(\mathbf{x})^\top\varphi(\mathbf{z})$ y kernel

Tomamos $\mathbf{z}=[1,4]^\top$ y comparamos el producto punto en el espacio de características con el valor directo del kernel.

In [3]:
z = np.array([1.0, 4.0])
phi_z = phi(z, c)

producto_phi = phi_x @ phi_z
producto_xz = x @ z
kernel = (producto_xz + c) ** 2

print("z =", z)
print("φ(z) =", np.round(phi_z, 4))
print()
print("φ(x)ᵀφ(z) =", round(float(producto_phi), 4))
print("xᵀz       =", round(float(producto_xz), 4))
print("K(x, z)   = (xᵀz + c)² =", round(float(kernel), 4))
print()
print("¿np.isclose(φ(x)ᵀφ(z), K(x, z))?", np.isclose(producto_phi, kernel))
if np.isclose(producto_phi, kernel):
    print("coinciden")

z = [1. 4.]
φ(z) = [ 1.     16.      5.6569  4.2426 16.9706  9.    ]

φ(x)ᵀφ(z) = 1.0
xᵀz       = -10.0
K(x, z)   = (xᵀz + c)² = 1.0

¿np.isclose(φ(x)ᵀφ(z), K(x, z))? True
coinciden


**Cálculo a mano de los 6 términos del producto punto.**

$$
\varphi(\mathbf{x}) \approx [4,\; 9,\; -8.4853,\; 8.4853,\; -12.7279,\; 9],
\qquad
\varphi(\mathbf{z}) \approx [1,\; 16,\; 5.6569,\; 4.2426,\; 16.9706,\; 9].
$$

Usando las formas exactas,
$$
\varphi(\mathbf{z}) = [1,\; 16,\; 4\sqrt{2},\; 3\sqrt{2},\; 12\sqrt{2},\; 9],
$$
y entonces
$$
\begin{aligned}
\varphi(\mathbf{x})^\top\varphi(\mathbf{z})
&= 4\cdot 1 + 9\cdot 16 + (-6\sqrt{2})(4\sqrt{2}) + (6\sqrt{2})(3\sqrt{2}) + (-9\sqrt{2})(12\sqrt{2}) + 9\cdot 9 \\
&= 4 + 144 - 48 + 36 - 216 + 81 \\
&= 1.
\end{aligned}
$$

Por otro lado, $\mathbf{x}^\top\mathbf{z} = 2\cdot 1 + (-3)\cdot 4 = 2-12 = -10$, así que
$$
K(\mathbf{x},\mathbf{z}) = (-10+9)^2 = (-1)^2 = 1.
$$

Por lo tanto, $\varphi(\mathbf{x})^\top\varphi(\mathbf{z}) = K(\mathbf{x},\mathbf{z}) = 1$: **coinciden**.

## 5. Verificación general para $d=3,5,10$

Para comprobar que la construcción de $\varphi$ no depende de $d=2$, repetimos la comparación con vectores aleatorios de dimensión $3$, $5$ y $10$. La semilla está fija para que el resultado sea reproducible.

In [4]:
rng = np.random.default_rng(0)  # semilla fija

print(f"{'d':>3} | {'φ(x)ᵀφ(z)':>14} | {'K(x,z)':>14} | {'np.allclose':>12}")
print("-" * 55)

for d in [3, 5, 10]:
    x_d = rng.normal(size=d)
    z_d = rng.normal(size=d)

    valor_phi = phi(x_d, c) @ phi(z_d, c)
    valor_kernel = (x_d @ z_d + c) ** 2
    coincide = np.allclose(valor_phi, valor_kernel)

    print(f"{d:>3} | {valor_phi:>14.4f} | {valor_kernel:>14.4f} | {str(coincide):>12}")
    assert coincide, f"Falla para d={d}"

print()
print("La construcción de φ funciona para cualquier dimensión probada.")

  d |      φ(x)ᵀφ(z) |         K(x,z) |  np.allclose
-------------------------------------------------------
  3 |        86.7790 |        86.7790 |         True
  5 |        81.7012 |        81.7012 |         True
 10 |       102.1041 |       102.1041 |         True

La construcción de φ funciona para cualquier dimensión probada.


## 6. Punto c) — Número de componentes de $\varphi$

Para un vector de dimensión $d$, $\varphi(\mathbf{x})$ tiene

$$
d \text{ cuadrados} + \frac{d(d-1)}{2} \text{ cruzados} + d \text{ lineales} + 1 \text{ constante}
= \frac{(d+1)(d+2)}{2}
$$

componentes. Por ejemplo: $d=2\Rightarrow 6$, $d=10\Rightarrow 66$, $d=100\Rightarrow 5151$ y $d=1000\Rightarrow 501501$. La tabla siguiente muestra también el desglose por tipo de término.

In [5]:
def n_componentes(d):
    """Cantidad de componentes de φ(x) para un vector de dimensión d."""
    return (d + 1) * (d + 2) // 2


# Verificación: la fórmula coincide con el largo real de φ(x) para d = 1..20.
rng = np.random.default_rng(0)
for d in range(1, 21):
    x_d = rng.normal(size=d)
    assert n_componentes(d) == len(phi(x_d, c)), f"Falla para d={d}"

print("Verificación OK: n_componentes(d) == len(phi(x, c)) para d = 1, ..., 20.")
print()

print(f"{'d':>6} | {'cuadrados':>10} | {'cruzados':>10} | {'lineales':>10} | {'constante':>10} | {'total':>10}")
print("-" * 72)

for d in [2, 10, 100, 1000]:
    cuadrados = d
    cruzados = d * (d - 1) // 2
    lineales = d
    constante = 1
    total = n_componentes(d)

    # La tabla también comprueba la descomposición del total.
    assert total == cuadrados + cruzados + lineales + constante
    print(f"{d:>6} | {cuadrados:>10} | {cruzados:>10} | {lineales:>10} | {constante:>10} | {total:>10}")

Verificación OK: n_componentes(d) == len(phi(x, c)) para d = 1, ..., 20.

     d |  cuadrados |   cruzados |   lineales |  constante |      total
------------------------------------------------------------------------
     2 |          2 |          1 |          2 |          1 |          6
    10 |         10 |         45 |         10 |          1 |         66
   100 |        100 |       4950 |        100 |          1 |       5151
  1000 |       1000 |     499500 |       1000 |          1 |     501501


## 7. Punto c) — Costo de calcular $\varphi(\mathbf{x})^\top\varphi(\mathbf{z})$ frente a $K(\mathbf{x},\mathbf{z})$

**Forma directa.** Hay que construir $\varphi(\mathbf{x})$ y $\varphi(\mathbf{z})$, cada uno con $N(d)=(d+1)(d+2)/2=\mathcal{O}(d^2)$ componentes, y luego multiplicarlos. El costo es $\mathcal{O}(d^2)$ en tiempo y también $\mathcal{O}(d^2)$ en memoria.

**Forma con el kernel.** Se calcula $\mathbf{x}^\top\mathbf{z}$ (producto punto de $d$ números), se suma $c$ y se eleva al cuadrado. El costo es $\mathcal{O}(d)$ en tiempo y $\mathcal{O}(1)$ de memoria adicional.

La medición siguiente es empírica y orientativa: incluye el costo de construir $\varphi$ en Python y depende de la máquina. Lo importante es la tendencia, no el valor exacto de cada tiempo. Para el método directo se usan más repeticiones cuando $d$ es chico y menos cuando $d$ es grande; para el kernel, que es muy barato, se usan muchas repeticiones siempre.

In [6]:
def producto_phi(x, z, c):
    """Forma directa: construir φ(x), φ(z) y hacer su producto punto."""
    return phi(x, c) @ phi(z, c)


def kernel_cuadratico(x, z, c):
    """Forma indirecta: evaluar K(x,z) sin construir φ."""
    return (x @ z + c) ** 2


def medir_tiempo(funcion, x, z, c, repeticiones):
    """Devuelve el tiempo promedio por llamada, en segundos."""
    # Una llamada de calentamiento para no medir la primera inicialización.
    funcion(x, z, c)

    inicio = time.perf_counter()
    for _ in range(repeticiones):
        funcion(x, z, c)
    fin = time.perf_counter()

    return (fin - inicio) / repeticiones


def repeticiones_directo(d):
    """Muchas repeticiones para d chico; menos para d grande."""
    n = n_componentes(d)
    return max(5, min(200, 3_000_000 // n))


# El kernel es muy barato (microsegundos), así que promediamos muchas llamadas.
REPS_KERNEL = 50_000

rng = np.random.default_rng(0)

print(f"{'d':>5} | {'N=len(φ)':>9} | {'reps φ':>7} | {'reps K':>7} | {'directo (s)':>13} | {'kernel (s)':>12} | {'cociente':>10}")
print("-" * 81)

for d in [10, 100, 500, 1000]:
    x_d = rng.normal(size=d)
    z_d = rng.normal(size=d)
    reps_directo = repeticiones_directo(d)

    t_directo = medir_tiempo(producto_phi, x_d, z_d, c, reps_directo)
    t_kernel = medir_tiempo(kernel_cuadratico, x_d, z_d, c, REPS_KERNEL)
    cociente = t_directo / t_kernel

    print(f"{d:>5} | {n_componentes(d):>9} | {reps_directo:>7} | {REPS_KERNEL:>7} | {t_directo:>13.3e} | {t_kernel:>12.3e} | {cociente:>10.1f}")

    d |  N=len(φ) |  reps φ |  reps K |   directo (s) |   kernel (s) |   cociente
---------------------------------------------------------------------------------


   10 |        66 |     200 |   50000 |     1.480e-04 |    1.362e-06 |      108.6


  100 |      5151 |     200 |   50000 |     5.500e-03 |    1.287e-06 |     4274.1


  500 |    125751 |      23 |   50000 |     2.400e-01 |    2.990e-06 |    80273.1


 1000 |    501501 |       5 |   50000 |     7.811e-01 |    3.144e-06 |   248429.2


**Lectura de la tabla.** El tiempo directo crece muy rápido con $d$: al pasar de $d=100$ a $d=1000$, el número de componentes se multiplica por casi $100$ y el tiempo directo también se acerca a ese factor. El tiempo del kernel se mantiene en el orden de los microsegundos: su costo teórico es $\mathcal{O}(d)$, pero para $d\le 1000$ pesa más el costo fijo de la llamada en Python que el producto punto. Por eso lo importante no es el valor exacto de cada tiempo, sino la tendencia: la brecha entre ambos métodos (la columna `cociente`) crece con $d$, y a $d=1000$ el método directo resulta cientos de miles de veces más lento en esta medición.

## 8. Conclusión

El kernel permite obtener el mismo número que se obtendría en un espacio de características grande, pero sin construir $\varphi(\mathbf{x})$ ni $\varphi(\mathbf{z})$. Eso ahorra tiempo y memoria: se pasa de $\mathcal{O}(d^2)$ a $\mathcal{O}(d)$. Por eso los SVM pueden trabajar con fronteras muy flexibles sin pagar el costo de la dimensión alta. El kernel gaussiano es el caso extremo: su $\varphi$ tiene infinitas componentes, así que no se puede construir; solo se puede usar evaluando $K(\mathbf{x},\mathbf{z})$. Esa es la idea central de los kernels: trabajar en dimensión alta (o infinita) a través de una función escalar.

## Anexo — Carga del dataset `.arff` (no forma parte del punto 1)

Este bloque es solo para dejar el dataset `Rice_Cammeo_Osmancik` en formato CSV para los puntos siguientes. No interviene en el punto 1 y requiere `scipy` y `pandas`. El código busca el archivo en la carpeta actual o en `csv/`, y guarda el CSV en la misma carpeta donde está el `.arff` (normalmente `csv/rice.csv`).

In [7]:
# Anexo: conversión del dataset .arff a CSV (no forma parte del punto 1).
from pathlib import Path

from scipy.io import arff
import pandas as pd

# Buscamos el archivo en la carpeta actual o en csv/.
ruta_arff = Path("Rice_Cammeo_Osmancik.arff")
if not ruta_arff.exists():
    ruta_arff = Path("csv") / "Rice_Cammeo_Osmancik.arff"

data, meta = arff.loadarff(ruta_arff)
df = pd.DataFrame(data)
df["Class"] = df["Class"].str.decode("utf-8")   # las etiquetas vienen como bytes

# Guardamos el CSV en la misma carpeta que el .arff (csv/rice.csv).
ruta_csv = ruta_arff.parent / "rice.csv"
df.to_csv(ruta_csv, index=False)

print("Archivo leído:", ruta_arff)
print("Archivo CSV generado:", ruta_csv)
print("Filas y columnas:", df.shape)
print(df.head())

Archivo leído: csv/Rice_Cammeo_Osmancik.arff
Archivo CSV generado: csv/rice.csv
Filas y columnas: (3810, 8)
      Area   Perimeter  Major_Axis_Length  Minor_Axis_Length  Eccentricity  \
0  15231.0  525.578979         229.749878          85.093788      0.928882   
1  14656.0  494.311005         206.020065          91.730972      0.895405   
2  14634.0  501.122009         214.106781          87.768288      0.912118   
3  13176.0  458.342987         193.337387          87.448395      0.891861   
4  14688.0  507.166992         211.743378          89.312454      0.906691   

   Convex_Area    Extent   Class  
0      15617.0  0.572896  Cammeo  
1      15072.0  0.615436  Cammeo  
2      14954.0  0.693259  Cammeo  
3      13368.0  0.640669  Cammeo  
4      15262.0  0.646024  Cammeo  
